In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import loguniform

warnings.filterwarnings("ignore")

In [ ]:
TRAIN_PATH = Path("final/train_mc.parquet")
TEST_PATH = Path("final/test_mc.parquet")
OUT_DIR = Path("models/xgboost")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "multiclass_xgboost.joblib"

RANDOM_STATE = 42
VERBOSE = 2

In [ ]:
train_mc = pd.read_parquet(TRAIN_PATH)
test_mc = pd.read_parquet(TEST_PATH)

X_train = train_mc.drop("Attack", axis=1)
y_train = train_mc["Attack"].astype(int)
X_test = test_mc.drop("Attack", axis=1)
y_test = test_mc["Attack"].astype(int)

NUM_CLASSES = int(y_train.nunique())
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"\nClass distribution (train):\n{y_train.value_counts().sort_index()}")

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

param_dist = {
    # Controls complexity and speed:
    'learning_rate': loguniform(0.01, 0.2),
    'n_estimators': [5, 10],             # same as LightGBM for fair comparison
    'max_depth': [4, 6, 8, 10],          # XGBoost's main complexity control (level-wise)

    # Controls stability and overfitting:
    'min_child_weight': [1, 5, 10],      # XGBoost's equivalent of min_child_samples
    'subsample': [0.7, 0.8, 1.0],        # row sampling per tree
    'colsample_bytree': [0.7, 0.8, 1.0], # feature sampling per tree
}

In [ ]:
xgb_base = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=NUM_CLASSES,
    tree_method='hist',          # change to 'gpu_hist' for GPU, or device='cuda'
    n_jobs=1,
    random_state=RANDOM_STATE,
    eval_metric='mlogloss',
    verbosity=0,
)

In [ ]:
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1_macro',
    cv=skf,
    verbose=2,
    random_state=RANDOM_STATE,
    n_jobs=1
)

print("Starting Randomized Search CV for XGBoost tuning...")
random_search.fit(X_train, y_train)
print("Tuning complete.")

In [ ]:
best_params = random_search.best_params_
print("="*50)
print("Best Hyperparameters Found:")
print(best_params)

best_score = random_search.best_score_
print(f"\nBest Mean F1-Score (macro) from CV: {best_score:.4f}")
print("="*50)

In [ ]:
best = random_search.best_estimator_

joblib.dump(best, OUT_FILE)
print("Saved best model to:", OUT_FILE)

In [ ]:
y_pred = best.predict(X_test)
y_pred_train = best.predict(X_train)

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1-weighted: {test_f1:.4f}")

In [ ]:
print("Classification Report (Train):")
train_cr = classification_report(y_train, y_pred_train)
print(train_cr)

print("\nClassification Report (Test):")
test_cr = classification_report(y_test, y_pred)
print(test_cr)

In [ ]:
report_content = (
    "#" * 50 + "\n"
    "CLASSIFICATION REPORT (TRAIN SET)\n"
    "#" * 50 + "\n"
    f"{train_cr}\n\n"
    "#" * 50 + "\n"
    "CLASSIFICATION REPORT (TEST SET)\n"
    "#" * 50 + "\n"
    f"{test_cr}\n"
)

REPORT_OUT_PATH = OUT_DIR / "classification_reports.txt"
with open(REPORT_OUT_PATH, 'w') as f:
    f.write(report_content)

print(f"Saved classification reports to: {REPORT_OUT_PATH}")

In [ ]:
def plot_classification_report(y_true, y_pred, title, out_dir):
    """Plot classification report heatmap with support shown as plain numbers per row."""
    report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    df = pd.DataFrame(report_dict).transpose()

    support = df['support'].fillna(0).astype(int)

    drop_rows = ['accuracy', 'macro avg', 'weighted avg', 'micro avg']
    df = df.drop(drop_rows, errors='ignore')

    df_metrics = df.drop(columns=['support'], errors='ignore').astype(float)

    plt.figure(figsize=(8, 4))
    ax = sns.heatmap(
        df_metrics,
        annot=True,
        cmap="YlGnBu",
        fmt=".3f",
        linewidths=.5,
        linecolor='black',
        cbar=True
    )

    plt.subplots_adjust(right=0.88)

    for y, cls in enumerate(df_metrics.index):
        sup_val = support.loc[cls]
        ax.text(
            df_metrics.shape[1] + 0.6,
            y + 0.5,
            str(sup_val),
            va='center',
            ha='left',
            fontsize=10,
            color='black'
        )

    ax.text(
        df_metrics.shape[1] + 0.6,
        -0.2,
        "support",
        va='bottom',
        ha='left',
        fontsize=10,
        color='black',
        fontweight='bold'
    )

    plt.title(f"Classification Report Heatmap ({title})")
    plt.ylabel("Class")
    plt.xlabel("Metrics")
    plt.tight_layout()

    out_path = out_dir / f"classification_report_{title.lower().replace(' ', '_')}.png"
    plt.savefig(out_path)
    print(f"Saved {title} plot to: {out_path}")
    plt.show()

In [ ]:
print("\n" + "="*50)
print("GENERATING CLASSIFICATION REPORT PLOTS")
print("="*50)

plot_classification_report(
    y_train,
    y_pred_train,
    'Train Set - XGBoost',
    OUT_DIR
)

plot_classification_report(
    y_test,
    y_pred,
    'Test Set - XGBoost',
    OUT_DIR
)